# Previsão de Atendimentos Individuais — Florianópolis (SC)

## Modelagem e Previsão de Séries Temporais com ARIMA e SARIMA

---

**Objetivo:** Prever o volume mensal de atendimentos médicos individuais na rede de atenção primária de Florianópolis, utilizando dados históricos e técnicas de modelagem de séries temporais.

**Série histórica:** 60 meses (janeiro/2019 – dezembro/2023)
**Horizonte de previsão:** 30 meses a partir do período de teste

**Metodologia aplicada:**
1. Análise exploratória da série temporal (ACF, PACF, teste de Ljung-Box)
2. Seleção automática de hiperparâmetros via `auto_arima` (busca *stepwise*)
3. Modelagem com **SARIMA** (componente sazonal) e **ARIMA** (não-sazonal)
4. Avaliação por métricas de acurácia (MAE, MSE, RMSE, MAPE, Coeficiente U₂ de Theil)
5. Diagnóstico de resíduos (teste de Durbin-Watson)

---

## 1. Configuração do Ambiente

In [ ]:
!pip install pmdarima

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Manipulação de dados e computação numérica
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ──────────────────────────────────────────────────────────────────────────────
# Visualização
# ──────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams

# ──────────────────────────────────────────────────────────────────────────────
# Estatística e diagnósticos
# ──────────────────────────────────────────────────────────────────────────────
import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox

# ──────────────────────────────────────────────────────────────────────────────
# Modelagem de séries temporais
# ──────────────────────────────────────────────────────────────────────────────
from pmdarima.arima import auto_arima

# ──────────────────────────────────────────────────────────────────────────────
# Métricas de avaliação
# ──────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

## 2. Parâmetros e Configurações do Projeto

> Todas as constantes que controlam o pipeline estão centralizadas nesta célula, facilitando a **reprodutibilidade** e a **manutenção** do projeto.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Caminho do dataset (ajustar conforme o ambiente de execução)
# ──────────────────────────────────────────────────────────────────────────────
DATASET_PATH = "/content/drive/My Drive/IC/VISITAxATENDIMENTOv6.xls"

# ──────────────────────────────────────────────────────────────────────────────
# Colunas de interesse no dataset
# ──────────────────────────────────────────────────────────────────────────────
COL_PERIODO     = "Período"
COL_ATENDIMENTO = "Atendimento Individual"
DATE_FORMAT     = "%d/%m/%Y"

# ──────────────────────────────────────────────────────────────────────────────
# Particionamento temporal (treino / teste)
# ──────────────────────────────────────────────────────────────────────────────
TRAIN_END_DATE  = "2022-06-30"
TEST_START_DATE = "2022-07-01"

# ──────────────────────────────────────────────────────────────────────────────
# Separação da previsão: período de validação vs. projeção futura
# ──────────────────────────────────────────────────────────────────────────────
FORECAST_VALIDATION_END = "09-30-2023"
FORECAST_FUTURE_START   = "10-1-2023"

# ──────────────────────────────────────────────────────────────────────────────
# Horizonte de previsão (em meses)
# ──────────────────────────────────────────────────────────────────────────────
N_FORECAST_PERIODS = 30

# ──────────────────────────────────────────────────────────────────────────────
# Parâmetros do auto_arima
# ──────────────────────────────────────────────────────────────────────────────
ARIMA_START_P   = 1
ARIMA_START_Q   = 1
ARIMA_MAX_P     = 6
ARIMA_MAX_Q     = 6
SEASONAL_PERIOD = 12    # periodicidade mensal (m = 12 meses)

# ──────────────────────────────────────────────────────────────────────────────
# Níveis de significância para intervalos de confiança
# ──────────────────────────────────────────────────────────────────────────────
ALPHA_95 = 0.05
ALPHA_80 = 0.20

## 3. Funções Auxiliares

### 3.1 Coeficiente U₂ de Theil

O coeficiente U₂ de Theil compara a acurácia do modelo de previsão contra uma **previsão ingênua** (*naïve forecast*), em que o valor futuro é assumido como igual ao valor imediatamente anterior.

| Valor de U₂ | Interpretação |
|:---:|:---|
| U₂ < 1 | Modelo **supera** a previsão ingênua |
| U₂ = 1 | Modelo equivalente à previsão ingênua |
| U₂ > 1 | Modelo **inferior** à previsão ingênua |

> **Referência:** Theil, H. (1966). *Applied Economic Forecasting*. North-Holland Publishing Company.

In [ ]:
def theil_u2(y_true, y_pred):
    """
    Calcula o coeficiente U2 de Theil para avaliação de modelos de previsão.

    Compara o erro relativo de previsão do modelo com o erro de uma
    previsão ingênua (naïve), onde y_hat(t) = y(t-1).

    Parameters
    ----------
    y_true : array-like
        Valores reais observados da série temporal.
    y_pred : array-like
        Valores previstos pelo modelo.

    Returns
    -------
    float
        Coeficiente U2 de Theil.
    """
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)
    N = len(y_true)

    # Numerador — erro relativo do modelo de previsão
    forecast_relative_error = (y_pred[1:] - y_true[1:]) / y_true[:-1]
    numerator = np.sqrt((1 / N) * np.sum(np.power(forecast_relative_error, 2)))

    # Denominador — erro relativo da previsão ingênua (random walk)
    naive_relative_error = (y_true[1:] - y_true[:-1]) / y_true[:-1]
    denominator = np.sqrt((1 / N) * np.sum(np.power(naive_relative_error, 2)))

    return numerator / denominator

## 4. Aquisição e Pré-processamento dos Dados

### 4.1 Conexão com o Google Drive (ambiente Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 4.2 Carregamento do Dataset

In [ ]:
dataset = pd.read_excel(DATASET_PATH)

In [ ]:
dataset

### 4.3 Construção e Tratamento da Série Temporal

Etapas de pré-processamento:
1. Seleção das colunas relevantes (`Período` e `Atendimento Individual`)
2. Conversão de tipos (datetime e numérico)
3. Reamostragem para frequência mensal (média)

In [ ]:
# Seleção das colunas de interesse
time_series = dataset[[COL_PERIODO, COL_ATENDIMENTO]].copy()

# Conversão de tipos
time_series[COL_PERIODO] = pd.to_datetime(time_series[COL_PERIODO], format=DATE_FORMAT)
time_series[COL_ATENDIMENTO] = pd.to_numeric(time_series[COL_ATENDIMENTO])

In [ ]:
# Extração da série bruta (antes da reamostragem)
X_base = time_series[COL_ATENDIMENTO]

In [ ]:
# Indexação temporal e reamostragem para frequência mensal (média)
time_series = time_series.set_index(COL_PERIODO).resample('M').mean()

In [ ]:
# Série temporal final utilizada na modelagem
X = time_series[COL_ATENDIMENTO]

In [ ]:
X

## 5. Análise Exploratória — Identificação de Padrões Temporais

A análise das funções de autocorrelação é fundamental para identificar:
- **Tendência** e **sazonalidade** na série
- A ordem dos componentes AR (autoregressivo) e MA (média móvel)
- A necessidade de diferenciação para alcançar estacionariedade

### 5.1 Função de Autocorrelação (ACF)

A ACF mede a correlação entre a série e suas defasagens (*lags*). Um decaimento lento sugere não-estacionariedade; picos periódicos indicam sazonalidade.

In [ ]:
plot_acf(X)
plt.show()

### 5.2 Função de Autocorrelação Parcial (PACF)

A PACF isola a correlação direta de cada defasagem, removendo efeitos intermediários. Auxilia na determinação da ordem $p$ do componente autoregressivo (AR).

In [ ]:
plot_pacf(X, method='ywm')
plt.show()

### 5.3 Teste de Ljung-Box

O teste de Ljung-Box avalia a presença de autocorrelação significativa nos dados até um determinado *lag*. A hipótese nula ($H_0$) é de que **não existe** autocorrelação.

- **p-valor < 0.05** → Rejeita $H_0$: há autocorrelação significativa (série não é ruído branco)
- **p-valor ≥ 0.05** → Não rejeita $H_0$: sem evidência de autocorrelação

In [ ]:
acorr_ljungbox(X, lags=[24])

## 6. Modelagem SARIMA (*Seasonal ARIMA*)

O modelo SARIMA estende o ARIMA clássico incorporando componentes sazonais, sendo especialmente adequado para séries temporais com padrões periódicos (e.g., sazonalidade mensal).

$$SARIMA(p, d, q)(P, D, Q)_{m}$$

onde:
- $(p, d, q)$: ordem dos componentes não-sazonais (AR, diferenciação, MA)
- $(P, D, Q)_m$: ordem dos componentes sazonais com período $m$

### 6.1 Seleção Automática de Hiperparâmetros (`auto_arima`)

O algoritmo `auto_arima` realiza uma busca *stepwise* sobre o espaço de hiperparâmetros, selecionando a combinação que minimiza o critério de informação AIC (*Akaike Information Criterion*).

In [ ]:
sarima_model = auto_arima(
    X,
    start_p=ARIMA_START_P, start_q=ARIMA_START_Q,
    max_p=ARIMA_MAX_P, max_q=ARIMA_MAX_Q,
    m=SEASONAL_PERIOD,
    start_P=0,
    seasonal=True,
    d=1, D=1,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

### 6.2 Particionamento dos Dados (Treino / Teste)

| Conjunto | Período |
|:---|:---|
| **Treino** | Início da série — junho/2022 |
| **Teste** | Julho/2022 — final da série |

In [ ]:
# Particionamento temporal
train = X.loc[:TRAIN_END_DATE]
test  = X.loc[TEST_START_DATE:]

In [ ]:
len(test)

### 6.3 Ajuste do Modelo e Geração de Previsões

In [ ]:
sarima_model.fit(train)

In [ ]:
# Geração de previsões para N_FORECAST_PERIODS meses à frente
sarima_future_forecast = sarima_model.predict(n_periods=N_FORECAST_PERIODS)

In [ ]:
sarima_future_forecast

In [ ]:
# Separação: período de validação vs. projeção futura
sarima_future_forecast_1 = sarima_future_forecast[:FORECAST_VALIDATION_END]
sarima_future_forecast_2 = sarima_future_forecast[FORECAST_FUTURE_START:]

In [ ]:
sarima_future_forecast_2

### 6.4 Visualização: Série Histórica vs. Previsão SARIMA

In [ ]:
sarima_future_forecast_2.plot(marker='', color='red', legend=True, label='Previsto')
X.plot(marker='', color='blue', legend=True, label='Real')

plt.show()

### 6.5 Intervalos de Confiança (80% e 95%)

Os intervalos de confiança são calculados assumindo distribuição normal para os erros de previsão, utilizando os valores críticos $z_{\alpha/2}$ e o desvio-padrão da série histórica.

In [ ]:
# Valores críticos da distribuição normal
z_critical_95 = stats.norm.ppf(1 - ALPHA_95 / 2)
z_critical_80 = stats.norm.ppf(1 - ALPHA_80 / 2)

# Cálculo dos intervalos de confiança
forecast_mean = sarima_future_forecast_2
forecast_std = np.std(X)

lower_bound_95 = forecast_mean - z_critical_95 * forecast_std
upper_bound_95 = forecast_mean + z_critical_95 * forecast_std

lower_bound_80 = forecast_mean - z_critical_80 * forecast_std
upper_bound_80 = forecast_mean + z_critical_80 * forecast_std

In [ ]:
sarima_future_forecast.index

In [ ]:
# Plotting
plt.figure(figsize=(12, 5))
X.plot(marker='', label='Previsão')
sarima_future_forecast_2.plot(marker='', label='Dados reais')


# Plotting confidence intervals
plt.fill_between(sarima_future_forecast_2.index, lower_bound_95, upper_bound_95, color='gray', alpha=0.2, label='Intervalo de Confiança (95%)')
plt.fill_between(sarima_future_forecast_2.index, lower_bound_80, upper_bound_80, color='blue', alpha=0.2, label='Intervalo de Confiança (80%)')
plt.title('Previsao SARIMA com Intervalo de Confianca')
plt.xlabel('Ano')
plt.ylabel('Numero de Atendimentos Individuais')
plt.legend(loc='upper left')

### 6.6 Métricas de Avaliação de Desempenho

| Métrica | Descrição |
|:---|:---|
| **MAE** | Erro Absoluto Médio — magnitude média dos erros |
| **MSE** | Erro Quadrático Médio — penaliza erros de maior magnitude |
| **RMSE** | Raiz do MSE — na mesma unidade dos dados originais |
| **MAPE** | Erro Percentual Absoluto Médio — erro relativo (%) |
| **TU** | Coeficiente U₂ de Theil — desempenho vs. previsão ingênua |

In [ ]:
# Calcular o Erro Absoluto Médio (MAE)
mae = mean_absolute_error(test,sarima_future_forecast_1)
print(f'MAE: {mae}')

# Calcular o Erro Quadrático Médio (MSE)
mse = mean_squared_error(test,sarima_future_forecast_1)
print(f'MSE: {mse}')

# Calcular a Raiz do Erro Quadrático Médio (RMSE)
rmse = np.sqrt(mse)
print(f'RMSE: {rmse}')

mape = mean_absolute_percentage_error(test,sarima_future_forecast_1)
print(f'MAPE: {mape}')

TU = theil_u2(test, sarima_future_forecast_1)
print(f'TU: {TU}')

### 6.7 Diagnóstico de Resíduos — Teste de Durbin-Watson

O teste de Durbin-Watson avalia a presença de **autocorrelação de primeira ordem** nos resíduos do modelo ajustado.

| Valor DW | Interpretação |
|:---:|:---|
| ≈ 2.0 | Sem autocorrelação nos resíduos (**ideal**) |
| < 2.0 | Autocorrelação **positiva** |
| > 2.0 | Autocorrelação **negativa** |

In [ ]:
model_fit = sarima_model.fit(train)

In [ ]:
durbin_watson(model_fit.resid())

## 7. Modelagem ARIMA (*AutoRegressive Integrated Moving Average*)

O modelo ARIMA é uma abordagem clássica para séries temporais **não-sazonais**. Diferentemente do SARIMA, não incorpora componentes sazonais explícitos.

$$ARIMA(p, d, q)$$

### 7.1 Seleção Automática de Hiperparâmetros

In [ ]:
arima_model = auto_arima(
    X,
    start_p=ARIMA_START_P, start_q=ARIMA_START_Q,
    max_p=ARIMA_MAX_P, max_q=ARIMA_MAX_Q,
    seasonal=False,
    d=1, D=1,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

### 7.2 Ajuste do Modelo e Geração de Previsões

In [ ]:
arima_model.fit(train)

In [ ]:
future_forecast_arima = arima_model.predict(n_periods=N_FORECAST_PERIODS)

In [ ]:
future_forecast_arima.index

In [ ]:
# Separação: período de validação vs. projeção futura
future_forecast_arima_1 = future_forecast_arima[:FORECAST_VALIDATION_END]
future_forecast_arima_2 = future_forecast_arima[FORECAST_FUTURE_START:]

In [ ]:
future_forecast_arima_2

### 7.3 Visualização: Série Histórica vs. Previsão ARIMA

In [ ]:
future_forecast_arima_2.plot(marker='', color='blue', legend=True, label='Previsto')
X.plot(marker='', color='red', label='Real', legend=True)

plt.show()

### 7.4 Intervalos de Confiança (80% e 95%)

In [ ]:
# Valores críticos da distribuição normal
z_critical_95 = stats.norm.ppf(1 - ALPHA_95 / 2)
z_critical_80 = stats.norm.ppf(1 - ALPHA_80 / 2)

# Cálculo dos intervalos de confiança
forecast_mean = future_forecast_arima_2
forecast_std = np.std(X)

lower_bound_95 = forecast_mean - z_critical_95 * forecast_std
upper_bound_95 = forecast_mean + z_critical_95 * forecast_std

lower_bound_80 = forecast_mean - z_critical_80 * forecast_std
upper_bound_80 = forecast_mean + z_critical_80 * forecast_std

In [ ]:
# Plotting
plt.figure(figsize=(12, 5))
X.plot(marker='', label='Previsão')
future_forecast_arima_2.plot(marker='', label='Dados reais')


# Plotting confidence intervals
plt.fill_between(future_forecast_arima_2.index, lower_bound_95, upper_bound_95, color='gray', alpha=0.2, label='Intervalo de Confiança (95%)')
plt.fill_between(future_forecast_arima_2.index, lower_bound_80, upper_bound_80, color='blue', alpha=0.2, label='Intervalo de Confiança (80%)')
plt.title('Previsao ARIMA com Intervalo de Confianca')
plt.xlabel('Ano')
plt.ylabel('Numero de Atendimentos Individuais')
plt.legend(loc='upper left')

### 7.5 Métricas de Avaliação de Desempenho

In [ ]:
# Calcular o Erro Absoluto Médio (MAE)
mae = mean_absolute_error(test,future_forecast_arima_1)
print(f'MAE: {mae}')

# Calcular o Erro Quadrático Médio (MSE)
mse = mean_squared_error(test,future_forecast_arima_1)
print(f'MSE: {mse}')

# Calcular a Raiz do Erro Quadrático Médio (RMSE)
rmse = np.sqrt(mse)
print(f'RMSE: {rmse}')

mape = mean_absolute_percentage_error(test,future_forecast_arima_1)
print(f'MAPE: {mape}')

TU = theil_u2(test, future_forecast_arima_1)
print(f'TU: {TU}')

### 7.6 Diagnóstico de Resíduos — Teste de Durbin-Watson

In [ ]:
model_fit = arima_model.fit(train)

In [ ]:
durbin_watson(model_fit.resid())

---

## 8. Considerações Finais

Este estudo aplicou dois modelos de séries temporais — **ARIMA** e **SARIMA** — para prever o volume mensal de atendimentos individuais na rede de atenção primária de Florianópolis (SC), utilizando uma série histórica de 60 meses (2019–2023).

**Critérios de avaliação empregados:**
- **Métricas de acurácia:** MAE, MSE, RMSE, MAPE e Coeficiente U₂ de Theil
- **Diagnóstico de resíduos:** Teste de Durbin-Watson para autocorrelação residual

A comparação entre os modelos permite identificar qual abordagem captura melhor os padrões subjacentes da série — especialmente a sazonalidade mensal, quando presente — fornecendo subsídios para o planejamento da capacidade de atendimento na rede municipal de saúde.